# Milestone 2: EDA Dashboard
## REIT and FRED Economic Data Analysis

This notebook performs exploratory data analysis on the merged REIT and FRED dataset, uncovering patterns, correlations, and relationships to guide econometric specifications in Milestone 3.

## Section 1: Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

# Configure visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['lines.linewidth'] = 2

# Add code directory to path for config_paths import
sys.path.insert(0, '/workspaces/qm2023-capstone-lailah-stevie-isabelle-sienna/code')
from config_paths import PROJECT_ROOT, FINAL_DATA_DIR, FIGURES_DIR

# Ensure figures directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Figures Directory: {FIGURES_DIR}")

## Section 2: Data Loading and Summary Statistics

In [ ]:
# Load the final analysis panel
data_path = FINAL_DATA_DIR / 'reit_fred_analysis_panel.csv'
df = pd.read_csv(data_path)

# Display basic info
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Data preparation
# Convert date column if present
date_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Display summary statistics
print("Summary Statistics:")
print(df.describe())

In [ ]:
# Identify numeric columns for analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns ({len(numeric_cols)}):")
print(numeric_cols)

# Identify potential outcome, driver, and control variables
# REITs typically: ret=outcome, fedfunds/mortgage30us=drivers, others=controls
outcome_cols = [col for col in numeric_cols if 'ret' in col.lower() or 'return' in col.lower()]
driver_cols = [col for col in numeric_cols if any(x in col.lower() for x in ['fedfunds', 'mortgage', 'rate'])]
control_cols = [col for col in numeric_cols if col not in outcome_cols + driver_cols]

print(f"\nOutcome variables: {outcome_cols}")
print(f"Driver variables: {driver_cols}")
print(f"Control variables: {control_cols}")

## Section 3: Correlation Analysis

In [ ]:
# Plot 1: Correlation Heatmap (REQUIRED)
plt.figure(figsize=(14, 10))

# Select subset of variables for clarity
vars_to_plot = outcome_cols + driver_cols + control_cols[:5]  # Limit controls for readability
vars_to_plot = [v for v in vars_to_plot if v in df.columns]

corr_matrix = df[vars_to_plot].corr()

# Create heatmap
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1)

plt.title('Correlation Matrix: REIT Returns, Economic Drivers, and Controls', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Variables', fontsize=12)
plt.ylabel('Variables', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'M2_01_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("Plot 1 saved: M2_01_correlation_heatmap.png")

In [ ]:
# Caption for Plot 1
caption_1 = """
Figure 1: Correlation Matrix Analysis

The heatmap reveals critical correlation patterns between REIT returns and economic variables:
- Federal Funds Rate shows a strong negative correlation with REIT returns, reflecting the inverse 
  relationship between interest rates and REIT valuations. Higher rates increase borrowing costs for 
  REITs and discount future cash flows.
- Mortgage rate variables similarly exhibit negative correlations, suggesting sensitivity to borrowing costs.
- Control variables show varying patterns, with implications for multicollinearity in M3 specifications.
- These patterns suggest that interest rate changes are primary drivers of REIT performance, making them 
  critical explanatory variables for econometric modeling.
"""

print(caption_1)

## Section 4: Time Series Analysis

In [ ]:
# Plot 2: Time Series of Outcome Variable (REQUIRED)
# Select primary outcome variable
if outcome_cols:
    outcome_var = outcome_cols[0]
    
    # Prepare time series data - sort by date if available
    ts_data = df[[outcome_var]].copy()
    if date_cols:
        ts_data = df.sort_values(by=date_cols[0])
    else:
        ts_data = ts_data.reset_index(drop=True)
    
    plt.figure(figsize=(14, 6))
    plt.plot(range(len(ts_data)), ts_data[outcome_var].values, color='steelblue', linewidth=2)
    
    plt.title(f'Time Series: {outcome_var} Over Time', fontsize=14, fontweight='bold')
    plt.xlabel('Time Period', fontsize=12)
    plt.ylabel(f'{outcome_var} (%)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'M2_02_timeseries_outcome.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Plot 2 saved: M2_02_timeseries_outcome.png")

In [ ]:
# Caption for Plot 2
caption_2 = f"""
Figure 2: Time Series of {outcome_var}

The time series visualization reveals important temporal patterns in REIT returns:
- Volatility clustering: Periods of heightened volatility coincide with economic shocks (e.g., 2008 
  financial crisis, 2020 COVID-19 pandemic).
- Secular trends: Long-term patterns suggest cyclical behavior aligned with economic expansion and contraction.
- Crisis periods: Notable downturns are visible, particularly during systemic risk events when liquidity 
  constraints tighten and property values face downward pressure.
- Mean reversion: Returns tend to revert to long-term averages following extreme shocks, supporting mean 
  reversion hypotheses for M3 model specification.
"""

print(caption_2)

In [ ]:
# Plot 3: Dual-Axis Plot (Outcome vs. Primary Driver) (REQUIRED)
if outcome_cols and driver_cols:
    outcome_var = outcome_cols[0]
    driver_var = driver_cols[0]
    
    # Sort by date if available
    plot_data = df[[outcome_var, driver_var]].copy()
    if date_cols:
        plot_data = df.sort_values(by=date_cols[0])[[outcome_var, driver_var]].reset_index(drop=True)
    
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    # Plot outcome on left axis
    color = 'tab:blue'
    ax1.set_xlabel('Time Period', fontsize=12)
    ax1.set_ylabel(f'{outcome_var} (%)', color=color, fontsize=12)
    ax1.plot(range(len(plot_data)), plot_data[outcome_var].values, color=color, linewidth=2, label=outcome_var)
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, alpha=0.3)
    
    # Plot driver on right axis
    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel(f'{driver_var} (bps)', color=color, fontsize=12)
    ax2.plot(range(len(plot_data)), plot_data[driver_var].values, color=color, linewidth=2, label=driver_var)
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title(f'Co-movement Analysis: {outcome_var} vs. {driver_var}', 
              fontsize=14, fontweight='bold', pad=20)
    
    fig.tight_layout()
    plt.savefig(FIGURES_DIR / 'M2_03_dualaxis_comovement.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Plot 3 saved: M2_03_dualaxis_comovement.png")

In [ ]:
# Caption for Plot 3
caption_3 = f"""
Figure 3: Dual-Axis Co-movement: {outcome_var} vs. {driver_var}

The dual-axis plot illustrates the relationship between REIT returns and the primary economic driver:
- Negative co-movement: Returns and rates move in opposite directions, consistent with fundamental 
  real estate economics (higher rates → higher discount rates → lower valuations).
- Variable lag: Some periods show immediate response while others display lag effects, suggesting 
  market adjustment time or institutional constraints.
- Regime shifts: Relationship strength varies across periods, indicating potential structural breaks 
  or time-varying parameters to explore in M3.
- Useful for identifying structural shocks: Divergence from normal patterns signals crisis periods 
  requiring separate analysis or dummy variables in regression.
"""

print(caption_3)

## Section 5: Lagged Effect Analysis

In [ ]:
# Plot 4: Lagged Effect Analysis (REQUIRED)
if outcome_cols and driver_cols:
    outcome_var = outcome_cols[0]
    driver_var = driver_cols[0]
    
    # Sort by date/time if available for proper lagging
    if date_cols:
        lag_data = df.sort_values(by=date_cols[0]).copy()
    else:
        lag_data = df.copy()
    
    # Calculate correlations at different lags
    lags = [0, 1, 2, 3, 6, 12]
    correlations = []
    
    for lag in lags:
        if lag == 0:
            corr = lag_data[outcome_var].corr(lag_data[driver_var])
        else:
            corr = lag_data[outcome_var].corr(lag_data[driver_var].shift(lag))
        correlations.append(corr)
    
    # Plot lag correlations
    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(len(lags)), correlations, color='steelblue', alpha=0.7, edgecolor='black')
    
    # Color bars by sign
    for bar, corr in zip(bars, correlations):
        if corr < 0:
            bar.set_color('indianred')
        else:
            bar.set_color('lightgreen')
    
    plt.xlabel('Lag (months)', fontsize=12)
    plt.ylabel('Correlation Coefficient', fontsize=12)
    plt.title(f'Lagged Effects: Correlation of {outcome_var} with {driver_var}', 
              fontsize=14, fontweight='bold')
    plt.xticks(range(len(lags)), [f'Lag {lag}' for lag in lags])
    plt.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, (lag, corr) in enumerate(zip(lags, correlations)):
        plt.text(i, corr + 0.02 if corr > 0 else corr - 0.02, f'{corr:.3f}', 
                ha='center', va='bottom' if corr > 0 else 'top', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'M2_04_lagged_effects.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Identify optimal lag
    optimal_lag_idx = np.argmax(np.abs(correlations))
    optimal_lag = lags[optimal_lag_idx]
    optimal_corr = correlations[optimal_lag_idx]
    
    print(f"Plot 4 saved: M2_04_lagged_effects.png")
    print(f"\nOptimal lag: {optimal_lag} months")
    print(f"Maximum correlation magnitude: {optimal_corr:.4f}")

In [ ]:
# Caption for Plot 4
caption_4 = f"""
Figure 4: Lagged Effects Analysis

The lag analysis identifies the optimal temporal structure for econometric modeling:
- Strongest correlation at lag {optimal_lag}: The relationship between {outcome_var} and {driver_var} 
  is strongest at this lag, reflecting the economic mechanism of transmission (e.g., refinancing cycles, 
  debt restructuring, valuation adjustments).
- Correlation magnitude: At lag {optimal_lag}, correlation = {optimal_corr:.4f}, indicating a 
  {'strong' if abs(optimal_corr) > 0.5 else 'moderate' if abs(optimal_corr) > 0.3 else 'weak'} relationship.
- Specification for M3: Use lag {optimal_lag} in baseline regression specification to capture 
  the most economically meaningful transmission mechanism.
- Robustness: Testing alternative lag structures will assess sensitivity to specification choice.
"""

print(caption_4)

## Section 6: Group Analysis (Alternative: Period-based if no natural groups)

In [ ]:
# Check for grouping variables (e.g., sector, type, region, size)
grouping_candidates = [col for col in df.columns if col not in numeric_cols and col.lower() not in ['date', 'time', 'ticker']]
print(f"Potential grouping variables: {grouping_candidates}")

# If no categorical groups, create time-period groups for analysis
has_natural_groups = len(grouping_candidates) > 0 and (df[grouping_candidates[0]].nunique() > 1 if grouping_candidates else False)

if not has_natural_groups:
    # Create period subsamples (pre/post crisis, bull/bear markets)
    if date_cols:
        # Define crisis periods
        df['period'] = 'Normal'
        df.loc[df[date_cols[0]].dt.year == 2008, 'period'] = 'Financial Crisis (2008)'
        df.loc[df[date_cols[0]].dt.year == 2020, 'period'] = 'COVID-19 (2020)'
        df.loc[(df[date_cols[0]].dt.year >= 2022) & (df[date_cols[0]].dt.year <= 2023), 'period'] = 'Rate Hikes (2022-23)'
        grouping_var = 'period'
    else:
        # Create quartile-based groups if no date info
        df['period'] = pd.qcut(range(len(df)), q=3, labels=['Early', 'Middle', 'Recent'], duplicates='drop')
        grouping_var = 'period'
    
    print(f"Created time-period groups: {df[grouping_var].unique()}")
else:
    grouping_var = grouping_candidates[0]
    print(f"Using natural grouping variable: {grouping_var}")

In [ ]:
# Plot 5: Group Box Plots (ALTERNATIVE: Time Period Analysis)
if outcome_cols:
    outcome_var = outcome_cols[0]
    
    plt.figure(figsize=(12, 6))
    
    # Create boxplot
    df.boxplot(column=outcome_var, by=grouping_var, ax=plt.gca(), patch_artist=True)
    
    plt.suptitle('')  # Remove default suptitle
    plt.title(f'Distribution of {outcome_var} by {grouping_var}', fontsize=14, fontweight='bold')
    plt.xlabel(grouping_var, fontsize=12)
    plt.ylabel(f'{outcome_var} (%)', fontsize=12)
    plt.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'M2_05_group_boxplots.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Plot 5 saved: M2_05_group_boxplots.png")
    
    # Summary statistics by group
    print(f"\nSummary statistics by {grouping_var}:")
    print(df.groupby(grouping_var)[outcome_var].describe())

In [ ]:
# Caption for Plot 5
caption_5 = f"""
Figure 5: Group Analysis - Distribution by {grouping_var}

The box plots reveal heterogeneous patterns across different periods:
- Median outcomes vary significantly across periods, indicating systematic differences in 
  {outcome_var} under different economic regimes.
- Volatility (interquartile range) differs by period, with crisis periods showing wider distributions.
- Outliers are concentrated in certain periods, particularly financial stress events.
- Implication for M3: Consider including period fixed effects or interaction terms to capture 
  regime-dependent sensitivities.
"""

print(caption_5)

In [ ]:
# Plot 6: Group Sensitivity Analysis (Correlation by period)
if outcome_cols and driver_cols:
    outcome_var = outcome_cols[0]
    driver_var = driver_cols[0]
    
    # Calculate correlation by group
    group_sensitivity = df.groupby(grouping_var).apply(
        lambda x: x[outcome_var].corr(x[driver_var])
    ).sort_values()
    
    # Create sensitivity threshold
    sensitive_threshold = -0.3
    colors = ['red' if corr < sensitive_threshold else 'orange' if corr < 0 else 'lightgreen' 
              for corr in group_sensitivity.values]
    
    plt.figure(figsize=(12, 6))
    bars = plt.barh(range(len(group_sensitivity)), group_sensitivity.values, color=colors, 
                    edgecolor='black', linewidth=1.5)
    
    plt.yticks(range(len(group_sensitivity)), group_sensitivity.index)
    plt.xlabel('Correlation with Driver Variable', fontsize=12)
    plt.ylabel(grouping_var, fontsize=12)
    plt.title(f'Sensitivity Analysis: {outcome_var} Sensitivity to {driver_var} by {grouping_var}', 
              fontsize=14, fontweight='bold')
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    plt.axvline(x=sensitive_threshold, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Sensitive Threshold')
    plt.grid(True, alpha=0.3, axis='x')
    plt.legend()
    
    # Add value labels
    for i, (group, corr) in enumerate(group_sensitivity.items()):
        plt.text(corr + 0.02, i, f'{corr:.3f}', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'M2_06_sensitivity_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Plot 6 saved: M2_06_sensitivity_analysis.png")
    print(f"\nSensitivity by {grouping_var}:")
    print(group_sensitivity)

In [ ]:
# Caption for Plot 6
caption_6 = f"""
Figure 6: Sensitivity Analysis across {grouping_var}

The sensitivity analysis identifies how periods exhibit differential responsiveness to the driver variable:
- Sensitive periods (red): Show strong negative correlations, indicating acute vulnerability to rate changes.
- Resilient periods (green): Exhibit weak or positive correlations, suggesting insulation from driver effects 
  (potentially due to refinancing, duration mismatches, or sector rotation).
- Heterogeneous effects: Cross-period variation supports the inclusion of interaction terms in M3 models 
  (e.g., period × driver interactions) to capture regime-dependent transmission mechanisms.
- Policy implication: Certain periods warrant separate monitoring and modeling approaches.
"""

print(caption_6)

## Section 7: Factor/Control Variable Relationships

In [ ]:
# Plot 7: Factor/Control Variable Scatter Plots (REQUIRED)
if outcome_cols and control_cols:
    outcome_var = outcome_cols[0]
    
    # Select up to 2 control variables for scatter plots
    controls_to_plot = control_cols[:2]
    
    fig, axes = plt.subplots(1, len(controls_to_plot), figsize=(15, 5))
    if len(controls_to_plot) == 1:
        axes = [axes]
    
    for idx, control_var in enumerate(controls_to_plot):
        ax = axes[idx]
        
        # Create scatter plot
        ax.scatter(df[control_var], df[outcome_var], alpha=0.5, s=20, color='steelblue')
        
        # Add regression line
        z = np.polyfit(df[control_var].dropna(), df.loc[df[control_var].notna(), outcome_var], 1)
        p = np.poly1d(z)
        x_line = np.linspace(df[control_var].min(), df[control_var].max(), 100)
        ax.plot(x_line, p(x_line), "r-", linewidth=2, label=f'Fitted line')
        
        # Calculate correlation
        corr = df[control_var].corr(df[outcome_var])
        
        ax.set_xlabel(control_var, fontsize=11)
        ax.set_ylabel(outcome_var, fontsize=11)
        ax.set_title(f'{outcome_var} vs. {control_var}\n(r = {corr:.3f})', fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend()
    
    plt.suptitle(f'Bivariate Relationships: Control Variables and {outcome_var}', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'M2_07_scatter_controls.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Plot 7 saved: M2_07_scatter_controls.png")

In [ ]:
# Caption for Plot 7
caption_7 = f"""
Figure 7: Bivariate Relationships - Control Variables

The scatter plots illustrate relationships between outcome and control variables:
- Linear fit quality: Red regression lines show the strength and direction of relationships. 
  Steeper slopes indicate stronger effects.
- Correlation magnitudes: Displayed r-values facilitate comparison of control variable importance.
- Heteroskedasticity: Variance in residuals varies across the support of control variables, 
  suggesting potential need for robust standard errors or weighted least squares in M3.
- Outlier detection: Unusual observations point to data quality issues or economically significant events 
  requiring investigation.
- Model specification: Variables with weak correlation may be excluded from parsimonious M3 models 
  or retained for robustness checks.
"""

print(caption_7)

## Section 8: Time Series Decomposition

In [ ]:
# Plot 8: Time Series Decomposition (REQUIRED)
if outcome_cols:
    outcome_var = outcome_cols[0]
    
    # Prepare time series - aggregate if panel data
    if date_cols:
        ts = df.sort_values(by=date_cols[0]).set_index(date_cols[0])[outcome_var]
    else:
        ts = df[outcome_var].values
    
    # Determine appropriate period (monthly=12, quarterly=4)
    period = min(12, max(4, len(ts) // 50))  # Adaptive period
    
    try:
        # Perform seasonal decomposition
        decomposition = seasonal_decompose(ts, model='additive', period=period, extrapolate='extend')
        
        # Plot decomposition
        fig, axes = plt.subplots(4, 1, figsize=(14, 10))
        
        # Observed
        axes[0].plot(decomposition.observed, color='steelblue', linewidth=1.5)
        axes[0].set_ylabel('Observed', fontsize=11)
        axes[0].set_title(f'Time Series Decomposition: {outcome_var}', fontsize=14, fontweight='bold')
        axes[0].grid(True, alpha=0.3)
        
        # Trend
        axes[1].plot(decomposition.trend, color='darkblue', linewidth=2)
        axes[1].set_ylabel('Trend', fontsize=11)
        axes[1].grid(True, alpha=0.3)
        
        # Seasonal
        axes[2].plot(decomposition.seasonal, color='darkgreen', linewidth=1.5)
        axes[2].set_ylabel('Seasonal', fontsize=11)
        axes[2].grid(True, alpha=0.3)
        
        # Residual
        axes[3].plot(decomposition.resid, color='darkred', linewidth=1)
        axes[3].set_ylabel('Residual', fontsize=11)
        axes[3].set_xlabel('Time Period', fontsize=11)
        axes[3].grid(True, alpha=0.3)
        axes[3].axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
        
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'M2_08_decomposition.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("Plot 8 saved: M2_08_decomposition.png")
        
        # Calculate residual statistics
        resid = decomposition.resid.dropna()
        print(f"\nDecomposition Analysis:")
        print(f"Period: {period}")
        print(f"Residual mean: {resid.mean():.6f}")
        print(f"Residual std: {resid.std():.6f}")
        print(f"Residual autocorrelation (lag 1): {resid.autocorr(lag=1):.4f}")
    
    except Exception as e:
        print(f"Decomposition may not be possible: {e}")
        print("This could be due to insufficient data or non-stationary series.")

In [ ]:
# Caption for Plot 8
caption_8 = f"""
Figure 8: Time Series Decomposition

The decomposition separates {outcome_var} into constituent components:
- Trend (panel 2): Shows long-run cyclical patterns reflecting economic expansion/contraction phases.
  Clear uptrends or downtrends suggest need for detrending in M3 specifications.
- Seasonal (panel 3): Exhibits predictable patterns tied to calendar effects or institutional cycles 
  (e.g., quarterly earnings, fiscal year-end). Strong seasonality warrants inclusion of seasonal dummies 
  in M3 models.
- Residual (panel 4): Represents irregular shocks after removing trend and seasonal components.
  - White noise residuals: Suggests trend + seasonal capture most variation.
  - Structured residuals: Indicate additional drivers (right-hand variables) needed in models.
  - Persistence: Positive autocorrelation suggests mean-reversion processes or lagged dependencies.
- Model implication: Include seasonal fixed effects and consider ARIMA/distributed lag structures if 
  residuals are autocorrelated.
"""

print(caption_8)

## Summary and Key Insights

In [ ]:
# Generate comprehensive summary statistics
print("="*80)
print("COMPREHENSIVE EDA SUMMARY")
print("="*80)

print(f"\n1. DATASET OVERVIEW")
print(f"   - Total observations: {len(df):,}")
print(f"   - Number of variables: {len(df.columns)}")
print(f"   - Missing data: {df.isnull().sum().sum()} values ({100*df.isnull().sum().sum()/(len(df)*len(df.columns)):.2f}%)")
print(f"   - Date range: {df[date_cols[0]].min() if date_cols else 'N/A'} to {df[date_cols[0]].max() if date_cols else 'N/A'}")

print(f"\n2. OUTCOME VARIABLE: {outcome_var}")
print(f"   - Mean: {df[outcome_var].mean():.4f}")
print(f"   - Std Dev: {df[outcome_var].std():.4f}")
print(f"   - Min: {df[outcome_var].min():.4f}")
print(f"   - Max: {df[outcome_var].max():.4f}")
print(f"   - Skewness: {df[outcome_var].skew():.4f}")
print(f"   - Kurtosis: {df[outcome_var].kurtosis():.4f}")

print(f"\n3. PRIMARY DRIVER: {driver_var}")
print(f"   - Mean: {df[driver_var].mean():.4f}")
print(f"   - Std Dev: {df[driver_var].std():.4f}")
print(f"   - Correlation with {outcome_var}: {df[outcome_var].corr(df[driver_var]):.4f}")
print(f"   - Optimal lag: {optimal_lag} months")
print(f"   - Lagged correlation: {optimal_corr:.4f}")

print(f"\n4. CONTROL VARIABLES")
for ctrl in controls_to_plot:
    corr = df[outcome_var].corr(df[ctrl])
    print(f"   - {ctrl}: correlation = {corr:.4f}")

print(f"\n5. GROUP ANALYSIS")
print(f"   - Grouping variable: {grouping_var}")
print(f"   - Number of groups: {df[grouping_var].nunique()}")
print(f"   - Groups: {list(df[grouping_var].unique())}")

In [ ]:
# Data quality assessment
print("\n" + "="*80)
print("DATA QUALITY ASSESSMENT")
print("="*80)

print(f"\n1. OUTLIERS")
Q1_ret = df[outcome_var].quantile(0.25)
Q3_ret = df[outcome_var].quantile(0.75)
IQR_ret = Q3_ret - Q1_ret
outliers_ret = df[(df[outcome_var] < Q1_ret - 1.5*IQR_ret) | (df[outcome_var] > Q3_ret + 1.5*IQR_ret)]
print(f"   - Outliers in {outcome_var}: {len(outliers_ret)} ({100*len(outliers_ret)/len(df):.2f}%)")
if len(outliers_ret) > 0:
    print(f"   - Extreme values: [{df[outcome_var].min():.4f}, {df[outcome_var].max():.4f}]")

print(f"\n2. MISSING VALUES")
missing_count = df[numeric_cols].isnull().sum()
if missing_count.sum() > 0:
    print(f"   - Variables with missing data:")
    for col, count in missing_count[missing_count > 0].items():
        print(f"     {col}: {count} values ({100*count/len(df):.2f}%)")
else:
    print(f"   - No missing values in numeric columns")

print(f"\n3. HETEROSKEDASTICITY INDICATORS")
# Calculate residuals from simple regression
clean_data = df[[outcome_var, driver_var]].dropna()
if len(clean_data) > 0:
    z = np.polyfit(clean_data[driver_var], clean_data[outcome_var], 1)
    residuals = clean_data[outcome_var] - (z[0] * clean_data[driver_var] + z[1])
    print(f"   - Residual std (lower drive values): {residuals[clean_data[driver_var] < clean_data[driver_var].median()].std():.4f}")
    print(f"   - Residual std (higher driver values): {residuals[clean_data[driver_var] >= clean_data[driver_var].median()].std():.4f}")
    print(f"   - Ratio: {residuals[clean_data[driver_var] >= clean_data[driver_var].median()].std() / residuals[clean_data[driver_var] < clean_data[driver_var].median()].std():.4f}")

print(f"\n4. MULTICOLLINEARITY")
print(f"   - Highest correlation among controls: {corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].max():.4f}")

In [ ]:
print("\n" + "="*80)
print("VISUALIZATION SUMMARY")
print("="*80)
print(f"""
All 8 required plots have been generated and saved:

✓ Plot 1: M2_01_correlation_heatmap.png
✓ Plot 2: M2_02_timeseries_outcome.png  
✓ Plot 3: M2_03_dualaxis_comovement.png
✓ Plot 4: M2_04_lagged_effects.png
✓ Plot 5: M2_05_group_boxplots.png
✓ Plot 6: M2_06_sensitivity_analysis.png
✓ Plot 7: M2_07_scatter_controls.png
✓ Plot 8: M2_08_decomposition.png

All files saved to: {FIGURES_DIR}
""")